# Battle 采样节点（rollout / eval）— Kaggle + cloudflared（单 cell 极简版）

这台云机**只采样本与跑评估**，不训练：起 `tools/agent/sampler-agent.ts`，把它的 HTTP 端口经
cloudflared 暴露成公网 URL，训练机（本机 hub / trainer）按 `rl-config.json` 的 `nodes` 把
rollout 局与 eval 局派过来。

| 步骤 | 怎么做 |
|---|---|
| 下载代码库 | `git clone --depth 1`（**全仓**；理由见 cell 内注释） |
| 装 bun | `curl -fsSL https://bun.sh/install | bash`（与 `battle-bc.ipynb` 同招） |
| 并发度 | `platform_utils.cpu_worker_slots()`（rollout/eval 的**唯一口径**） |
| 公网入口 | `cloudflared tunnel --url http://localhost:<port>` |
| 干活 | `bun tools/agent/sampler-agent.ts --port P --workers N --cache-mb M` |
| 保活 | 心跳打印 + agent 崩溃自动重启 + 隧道换名检测 |

**用法**：填 `CFG.repo`（+ 私有仓的 `github_token`）→ Run All → 等它打印 ★ 隧道 URL 与
`authKey` → 把这两个值写进本机 `nn-training/rl-config.json` 的 `nodes`：

```json
{ "nodes": [ { "id": "kaggle-rollout", "url": "<隧道URL>", "authKey": "<authKey>", "concurrency": 0 } ] }
```

（`concurrency` 不填 = 不额外限流；节点门要 `ping` 通且 `evalSupport` 为真 —— 见
`rl/batch_eval.py:101`。）

**停止**：■ 中断本 cell。Kaggle 单次会话上限按 `max_session_hours`（缺省 9h）。

In [ ]:
# @title Battle 采样节点 —— 填 CFG.repo（+ 私有仓 token）后直接 Run
import json, os, re, subprocess, sys, threading, time, urllib.request
from pathlib import Path

CFG = {
    # ── 代码库 ──────────────────────────────────────────────────────────
    # 采样节点要的是 **tools/（TS 导出器）+ src/（wasm 内核）+ nn-training/**：
    # hub 的 `GET /code` 只有 nn-training 下的 .py/.jsonc（remote/hub_client.py:385-394），
    # 而 TS 运行时在 ts_code.zip（src+tools，:456）且只随具体 job 下发 —— 采样节点没有
    # job，所以直接 clone 全仓是唯一省事的路（Kaggle 有外网）。
    "repo": "https://github.com/<owner>/battle2.git",
    "ref": "main",
    "github_token": "",        # 私有仓 PAT；留空则读环境变量 GITHUB_TOKEN
    "code_dir": "/kaggle/working/battle2",
    # ── agent ───────────────────────────────────────────────────────────
    "agent_port": 8443,
    "workers": 0,             # 0 = 自动（platform_utils.cpu_worker_slots；>0 完全照填的数）
    "cache_mb": 2048,         # 结果缓存上限（同键重入直接回放）
    "agent_extra": [],        # 例 ["--no-node"] = 采样子进程强制 bun（A/B 与回滚用）
    "bun_install_deps": False,  # agent 零第三方依赖；勾 True 才跑 bun install
    # ── 会话与保活 ───────────────────────────────────────────────────────
    "max_session_hours": 9,   # Kaggle 单次会话上限；到点干净收线
    "restart_limit": 5,       # agent 崩溃自动重启上限（超过就停cell，别无限重启）
    "keepalive_every_sec": 120,
    "cloudflared_path": "",   # 留空 = 自动找/自动装
}

def _log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] [rollout-node] {msg}", flush=True)

def _sh(cmd, *, cwd=None, timeout=None):
    try:
        return subprocess.run(cmd, cwd=cwd, capture_output=True, text=True, timeout=timeout)
    except (OSError, subprocess.SubprocessError) as e:
        class _R:
            returncode = 127
            stdout = ''
            stderr = str(e)
        return _R()

# ── 保活：Kaggle 一段时间无输出会回收实例 —— 打印本身就是心跳 ──────────
# （与 battle.cloudflared.ipynb / battle.offline.ipynb 同一招；Colab 那个分支要
#  点连接按钮，Kaggle 只要别静默。）
_stop = threading.Event()

def _keepalive():
    n = 0
    every = max(30, int(CFG['keepalive_every_sec']))
    while not _stop.is_set():
        print(f"[{time.strftime('%H:%M:%S')}] [rollout-node] [keepalive] alive ({n * every // 60} min)", flush=True)
        n += 1
        _stop.wait(every)

threading.Thread(target=_keepalive, daemon=True, name='keepalive').start()
_log('Keepalive 已启动')

# ── 1. 代码库 ───────────────────────────────────────────────────────────
code_dir = Path(CFG['code_dir'])
repo = str(CFG['repo'] or '').strip()
ref = str(CFG['ref'] or 'main').strip() or 'main'
if (not repo) or '<owner>' in repo:
    raise SystemExit('[rollout-node] FATAL: CFG.repo 必填（形如 https://github.com/<owner>/battle2.git）')
_tok = str(CFG['github_token'] or '').strip() or os.environ.get('GITHUB_TOKEN', '')
_url = repo
if _tok and repo.startswith('https://'):
    _url = repo.replace('https://', f'https://{_tok}@')   # 私有仓；只在内存里拼，不落日志
code_dir.parent.mkdir(parents=True, exist_ok=True)
if (code_dir / '.git').is_dir():
    _sh(['git', '-C', str(code_dir), 'fetch', '--depth', '1', 'origin', ref], timeout=600)
    _sh(['git', '-C', str(code_dir), 'checkout', '--force', ref], timeout=300)
    _sh(['git', '-C', str(code_dir), 'reset', '--hard', f'origin/{ref}'], timeout=300)
else:
    r = _sh(['git', 'clone', '--depth', '1', '--branch', ref, _url, str(code_dir)], timeout=900)
    if r.returncode != 0:
        _log('git clone 失败（私有仓请填 github_token / Kaggle secret GITHUB_TOKEN）：')
        _log(r.stderr[-600:])
        raise SystemExit(2)
_head = _sh(['git', '-C', str(code_dir), 'rev-parse', '--short', 'HEAD']).stdout.strip()
_log(f'代码库就绪: {code_dir}  commit={_head}  ref={ref}')

# ── 2. bun ──────────────────────────────────────────────────────────────
bun = ''
for cand in ('bun', '/root/.bun/bin/bun', str(Path.home() / '.bun' / 'bin' / 'bun')):
    if _sh([cand, '--version'], timeout=120).returncode == 0:
        bun = cand
        break
if not bun:
    _log('安装 bun…')
    _sh(['bash', '-lc', 'curl -fsSL https://bun.sh/install | bash'], timeout=600)
    if _sh(['/root/.bun/bin/bun', '--version'], timeout=120).returncode == 0:
        bun = '/root/.bun/bin/bun'
if not bun:
    raise SystemExit('[rollout-node] FATAL: bun 不可用（也装不上）')
_bunv = _sh([bun, '--version']).stdout.strip()
_log(f'bun: {bun} {_bunv}')
_bunbin = str(Path(bun).parent)
if _bunbin and _bunbin != '.' and _bunbin not in os.environ.get('PATH', ''):
    os.environ['PATH'] = _bunbin + os.pathsep + os.environ.get('PATH', '')
if CFG['bun_install_deps']:
    r = _sh([bun, 'install', '--frozen-lockfile'], cwd=str(code_dir), timeout=900)
    _log(f'bun install rc={r.returncode}（agent 零第三方依赖，失败不致命）')

# ── 3. 并发度：与 rollout/eval 同一口径 ─────────────────────────────────
# 核数走 platform_utils.effective_cores（容器 cgroup 配额/亲和掩码 > 宿主机裸数：
# 容器里 os.cpu_count() 报的是宿主机的核数，Kaggle 实测 224 而配额只给 96——按宿主机数
# 派并发就是 2.3× 超订）；槽位用 cpu_worker_slots（≤4 核全给，否则 max(cores-2, cores*0.8)）：云机上
# rollout 与 eval 交替跑，两边都满配、谁在跑谁用满，只给补传/日志留两三个核。
cores = os.cpu_count() or 1   # 仅当 platform_utils 拉不起来时的兜底（宿主口径）
sys.path.insert(0, str(code_dir / 'nn-training'))
try:
    from platform_utils import cpu_worker_slots, effective_cores
    cores = effective_cores()
    slots = cpu_worker_slots(cores)
    _how = 'platform_utils.effective_cores + cpu_worker_slots'
except Exception as _e:
    slots = (cores if cores <= 4 else max(1, min(cores, max(cores - 2, int(cores * 0.8)))))   # 同规则内联回退
    _how = f'内联回退（{type(_e).__name__}）'
workers = int(CFG['workers']) if int(CFG['workers'] or 0) > 0 else slots
_log(f'CPU cores={cores} → workers={workers}（{_how}）')

# ── 4. cloudflared 隧道 ─────────────────────────────────────────────────
port = int(CFG['agent_port'])
cf = str(CFG['cloudflared_path'] or '').strip()
if not cf:
    _w = _sh(['bash', '-lc', 'command -v cloudflared || true'], timeout=60).stdout.strip()
    cf = _w.splitlines()[-1].strip() if _w else ''
if not cf:
    cf = '/usr/local/bin/cloudflared'
    _log('安装 cloudflared…')
    r = _sh(['curl', '-fsSL',
             'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
             '-o', cf], timeout=180)
    if r.returncode == 0:
        os.chmod(cf, 0o755)
_cflog = Path('/tmp/rollout-cloudflared.log')
cfp = subprocess.Popen([cf, 'tunnel', '--url', f'http://localhost:{port}',
                        '--logfile', str(_cflog)],
                       stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

_URL_RE = re.compile(r'https://[a-z0-9-]+\.trycloudflare\.com')

def _tunnel_url():
    """隧道 URL（取日志里**最后**一个：重连会换名，旧的不能再报）。

    quick tunnel 每次重连都换域名 —— 换名后还拿旧 URL 填 rl-config，训练机会整天撞
    HTTP 530（事故教训）。所以这里每次都重读日志，且变化时在监护循环里重新打印。
    """
    try:
        txt = _cflog.read_text(encoding='utf-8', errors='replace')
    except OSError:
        return ''
    m = _URL_RE.findall(txt)
    return m[-1] if m else ''

url, t0 = '', time.time()
while time.time() - t0 < 90 and not url:
    url = _tunnel_url()
    if url:
        break
    if cfp.poll() is not None:
        break
    time.sleep(2)
if not url:
    _log(f'⚠ 没拿到隧道 URL —— 下面是 {_cflog} 尾部：')
    try:
        for ln in _cflog.read_text(encoding='utf-8', errors='replace').splitlines()[-25:]:
            _log('  | ' + ln)
    except OSError:
        pass

# ── 5. sampler-agent ────────────────────────────────────────────────────
# 入口在 tools/agent/（旧注释里的 tools/dist/ 是历史路径，两个都探，先到先得）。
entry = None
for rel in ('tools/agent/sampler-agent.ts', 'tools/dist/sampler-agent.ts'):
    p = code_dir / rel
    if p.is_file():
        entry = p
        break
if entry is None:
    raise SystemExit(f'[rollout-node] FATAL: 找不到 sampler-agent.ts（{code_dir / "tools"}）')
_agentlog = Path('/tmp/rollout-agent.log')
_auth_path = entry.parent / 'agent.auth'   # agent 首启生成并打印一次；运维复制到 rl-config


def _start_agent():
    cmd = [bun, str(entry), '--port', str(port), '--workers', str(workers),
           '--cache-mb', str(int(CFG['cache_mb']))] + [str(a) for a in CFG['agent_extra']]
    fh = open(_agentlog, 'a')
    _log('agent 启动: ' + ' '.join(cmd))
    return subprocess.Popen(cmd, cwd=str(code_dir), stdout=fh, stderr=subprocess.STDOUT)


agent = _start_agent()

def _wait_auth(timeout=90):
    t = time.time()
    while time.time() - t < timeout:
        try:
            v = _auth_path.read_text(encoding='utf-8').strip()
        except OSError:
            v = ''
        if v:
            return v
        if agent.poll() is not None:
            return ''
        time.sleep(2)
    return ''


def _ping(auth='', timeout=5):
    """本地探活。/v1/ping **要鉴权**（sampler-agent.ts:1258-1260，只有 /pool 公开）
    —— 不带 Bearer 一律 401，监护循环会误判成「没起来」。"""
    try:
        req = urllib.request.Request(f'http://127.0.0.1:{port}/v1/ping')
        if auth:
            req.add_header('Authorization', f'Bearer {auth}')
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return json.loads(r.read().decode('utf-8'))
    except Exception:
        return None


auth = _wait_auth()
ping = None
t0 = time.time()
while time.time() - t0 < 60 and ping is None:
    ping = _ping(auth)
    if ping is None:
        time.sleep(2)
if ping is None:
    _log('agent 30–60s 未就绪 —— 下面是 agent 日志尾部（真因通常在这）：')
    try:
        for ln in _agentlog.read_text(encoding='utf-8', errors='replace').splitlines()[-40:]:
            _log('  | ' + ln)
    except OSError:
        pass

_log('=' * 68)
_log(f'★ 隧道 URL : {url or "(未取到)"}')
_log(f'★ authKey  : {auth or "(未生成)"}')
_log(f'  workers  : {workers} / cores {cores}   bun {_bunv}   codeHash={str((ping or {}).get("codeHash", ""))[:12]}')
_log('写进本机 nn-training/rl-config.json：')
_log('  {"nodes":[{"id":"kaggle-rollout","url":"%s","authKey":"%s"}]}' % (url, auth))
_log('=' * 68)

# ── 6. 监护：agent 死了重启 / 隧道换名重报 / 会话到点收线 ────────────────
deadline = time.time() + max(0.0, float(CFG['max_session_hours'])) * 3600
restarts, last_url, tick, miss = 0, url, 0, 0
try:
    while True:
        if time.time() > deadline:
            _log('会话上限到点 —— 收线')
            break
        if _stop.is_set():
            break
        # 健康判定：**进程在 ≠ 服务在**（端口还活着但 HTTP 死掉是最阴的一种）——
        # 每 60s 探一次 /v1/ping，连续 3 次不通才判死（单次抖动不该重启）。
        _dead, _why = False, ''
        if agent.poll() is not None:
            _dead, _why = True, f'rc={agent.returncode}'
        else:
            tick += 1
            if tick % 6 == 0:
                if _ping(auth) is None:
                    miss += 1
                    if miss >= 3:
                        _dead, _why = True, 'HTTP 探活连续 3 次失败'
                else:
                    miss = 0
        if _dead:
            if restarts >= int(CFG['restart_limit']):
                _log(f'agent 已重启 {restarts} 次仍不健康 —— 停止（看 {_agentlog}）')
                break
            restarts += 1
            _log(f'agent 判定死亡（{_why}）—— 第 {restarts} 次重启')
            try:
                if agent.poll() is None:
                    agent.terminate()
                    agent.wait(timeout=10)
            except Exception:
                pass
            agent = _start_agent()
            miss = 0
            a2 = _wait_auth(30)
            if a2 and a2 != auth:
                auth = a2
                _log(f'★ authKey 变了: {auth}')
        cur = _tunnel_url()
        if cur and cur != last_url:
            last_url = cur
            _log(f'★ 隧道换名（quick tunnel 重连）：{cur} —— rl-config 里的 url 要同步改')
        time.sleep(10)
except KeyboardInterrupt:
    _log('收到中断')
finally:
    _stop.set()
    for p, name in ((agent, 'agent'), (cfp, 'cloudflared')):
        try:
            if p.poll() is None:
                p.terminate()
                try:
                    p.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    p.kill()
                _log(f'{name} 已停止')
        except Exception:
            pass
_log('采样节点结束')